# Exploring in Cloud Filtering in GEE 

## Introduction

This notebook is designed to explore cloud masking strategies for Sentinel-2 and Landsat8 for the guyana coastline and seek to understand how to get cloudless images to feed into ML/DL models for extent map classification. The resources helped to assist in this notebook include: 
- [Claude Opus 4.8](https://claude.ai/share/8f5150e4-c297-4994-88e8-0272355124af)
- [Google Earth Engine Tutorial](https://developers.google.com/earth-engine/tutorials/community/sentinel-2-s2cloudless)
- [Gemini 3.5 Flash](https://gemini.google.com/) 

In [10]:
import ee
import geemap
import matplotlib.pyplot as plt 

ee.Authenticate()
ee.Initialize(project='irp-zu')

### Using bit shifting to mask clouds in Sentinel-2 (reduces most of the clouds but still not cloudless)

In [11]:
coastline_coords = [[-59.755689785228235, 8.126814578766439],
          [-57.833082363353235, 5.607387745118143],
          [-57.146881951761834, 5.898364580242008],
          [-57.009580630573, 6.02949062794878],
          [-56.888703240824334, 6.15511611857417],
          [-56.493195428324334, 6.624592088692309],
          [-56.416291131449334, 7.785376639618971],
          [-58.45482139063248, 9.543371065466095],
          [-59.98741416406998, 8.545208575574712]]

aoi_polygon = ee.Geometry.Polygon([coastline_coords])
aoi_polygon

start_date = "2020-01-01"
end_date = "2020-12-31"

# https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED#colab-python

def mask_s2_clouds(image):
  """Masks clouds in a Sentinel-2 image using the QA band.

  Args:
      image (ee.Image): A Sentinel-2 image.

  Returns:
      ee.Image: A cloud-masked Sentinel-2 image.
  """
  qa = image.select('QA60')

  # Bits 10 and 11 are clouds and cirrus, respectively.
  cloud_bit_mask = 1 << 10
  cirrus_bit_mask = 1 << 11

  # Both flags should be set to zero, indicating clear conditions.
  mask = (
      qa.bitwiseAnd(cloud_bit_mask)
      .eq(0)
      .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
  )

  return image.updateMask(mask).divide(10000)


sentinel = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi_polygon)
    .filterDate(start_date, end_date)
    # Pre-filter to get less cloudy granules.
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 100))
    .map(mask_s2_clouds)
)

visualization = {
    'min': 0.0,
    'max': 0.3,
    'bands': ['B4', 'B3', 'B2'],
}

m = geemap.Map()
m.center_object(aoi_polygon, 7)
m.add_layer(sentinel.median(), visualization, 'RGB')
m

Map(center=[7.625000281267503, -58.02523831094688], controls=(WidgetControl(options=['position', 'transparent_…

### Checking the number of images available in 2020 and the number of images with <5% cloud cover

In [12]:
# 1. Get the TOTAL number of images available for your AOI and year
total_collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi_polygon)
    .filterDate(start_date, end_date)
)

print("Total images available in 2020:", total_collection.size().getInfo())


# 2. Get the number of strictly "cloud-free" images (e.g., less than 5% cloud cover)
cloud_free_collection = total_collection.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5))

print("Strictly cloud-free images (<5% clouds):", cloud_free_collection.size().getInfo())


# 3. Apply your masking function to the full collection for your map
sentinel = total_collection.map(mask_s2_clouds)

# ... rest of your geemap visualization code ...

Total images available in 2020: 2031
Strictly cloud-free images (<5% clouds): 164


### Using the cloud probability dataset to generate cloudless images

In [13]:
# https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_CLOUD_PROBABILITY

# Look at this cloudfree tutorial: https://developers.google.com/earth-engine/tutorials/community/sentinel-2-s2cloudless

s2_sr = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
s2_clouds = ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')

max_cloud_probability = 50

def maskClouds(img):
  clouds = ee.Image(img.get('cloud_mask')).select('probability')
  is_not_cloud = clouds.lt(max_cloud_probability)
  return img.updateMask(is_not_cloud)


# The masks for the 10m bands sometimes do not exclude bad data at
# scene edges, so we apply masks from the 20m and 60m bands as well.
# Example asset that needs this operation:
# COPERNICUS/S2_CLOUD_PROBABILITY/20190301T000239_20190301T000238_T55GDP
def maskEdges(s2_img):
  return s2_img.updateMask(
      s2_img.select('B8A').mask().updateMask(s2_img.select('B9').mask())
  )


# Filter input collections by desired data range and region.
criteria = ee.Filter.And(
    ee.Filter.bounds(aoi_polygon), ee.Filter.date(start_date, end_date)
)
s2_sr = s2_sr.filter(criteria).map(maskEdges)
s2_clouds = s2_clouds.filter(criteria)

# Join S2 SR with cloud probability dataset to add cloud mask.
s2_sr_with_cloud_mask = ee.Join.saveFirst('cloud_mask').apply(
    primary=s2_sr,
    secondary=s2_clouds,
    condition=ee.Filter.equals(
        leftField='system:index', rightField='system:index'
    ),
)

# 1. Convert the joined list back into a clean ImageCollection
final_collection = ee.ImageCollection(s2_sr_with_cloud_mask)

# 2. Count the images and print the result using .getInfo()
image_count = final_collection.size().getInfo()

print(f"Total number of images used to create this composite for 2020: {image_count}")

truly_clear_collection = s2_sr.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5))
clear_count = truly_clear_collection.size().getInfo()

print(f"Out of {image_count} total images, only {clear_count} were naturally cloudless (<5% clouds).")

s_2_cloud_masked = (
    ee.ImageCollection(s2_sr_with_cloud_mask).map(maskClouds).median()
)

rgb_vis = {'min': 0, 'max': 3000, 'bands': ['B4', 'B3', 'B2']}

m = geemap.Map()
m.center_object(aoi_polygon, 7)
m.add_layer(
    s_2_cloud_masked,
    rgb_vis,
    'S2 SR masked at ' + str(max_cloud_probability) + '%',
    True,
)
m

Total number of images used to create this composite for 2020: 2031
Out of 2031 total images, only 164 were naturally cloudless (<5% clouds).


Map(center=[7.625000281267503, -58.02523831094688], controls=(WidgetControl(options=['position', 'transparent_…

### Applying bit shifting cloud mask to Landsat8 (shows significant holes)

In [14]:
# apply proper scale factors

# Initialize the map
m = geemap.Map()

def mask_l8_clouds(image):
    """Masks clouds and shadows in Landsat 8 Surface Reflectance data."""
    # Bit 3 = Cloud Shadow, Bit 4 = Cloud
    qa = image.select('QA_PIXEL')
    
    # Both flags should be 0 (indicating clear conditions)
    cloud_shadow_mask = 1 << 3
    cloud_mask = 1 << 4

    # Extract Cloud Confidence (Bits 8-9)
    # Shifting right by 8 bits isolates the confidence value (0 to 3)
    cloud_confidence = qa.rightShift(8).bitwiseAnd(3)
    
    # Combine masks: pixels must have NEITHER cloud nor shadow
    mask = (
        qa.bitwiseAnd(cloud_shadow_mask).eq(0)
        .And(qa.bitwiseAnd(cloud_mask).eq(0))
        .And(cloud_confidence.lt(2))
    )
    
    # Apply mask and scale the optical bands (SR bands are scaled by 0.0000275 + -0.2)
    optical_bands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    
    # Replace original optical bands with scaled and masked versions
    return image.addBands(optical_bands, overwrite=True).updateMask(mask)


# 1. Load and filter the Landsat 8 Surface Reflectance Tier 1 collection
landsat_collection = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .filterBounds(aoi_polygon)
    .filterDate(start_date, end_date)
)

# 2. Count the total images in the stack
landsat_count = landsat_collection.size().getInfo()
print(f"Total Landsat 8 images available for 2020: {landsat_count}")

# 3. Map the cloud mask and collapse into a single median image
landsat_masked = landsat_collection.map(mask_l8_clouds).median()

# 4. Clip to your coastal region of interest
# landsat_composite = landsat_masked.clip(aoi_polygon)

# Instead of: landsat_collection.map(mask_l8_clouds).median()
# Use a Quality Mosaic based on NDVI (Vegetation Index)

def add_ndvi(image):
    ndvi = image.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    return image.addBands(ndvi)

# This tells Earth Engine: For every pixel coordinate, look through all 2020 
# images and select the pixel from the day where the vegetation was highest/greenest. 
# Since clouds have terrible NDVI, this completely obliterates them.
landsat_composite = landsat_collection.map(mask_l8_clouds).map(add_ndvi).qualityMosaic('NDVI').clip(aoi_polygon)

# Landsat 8 True Color Bands: B4 (Red), B3 (Green), B2 (Blue)
# Note: Because of the scaling factor applied in the mask function, 
# reflectance values range from 0.0 to 1.0 (0.3 is a good max for land/vegetation)
landsat_vis = {
    'min': 0.0,
    'max': 0.3,
    'bands': ['SR_B4', 'SR_B3', 'SR_B2']
}

# 5. Display the result
m.center_object(aoi_polygon, 7)
m.add_layer(landsat_composite, landsat_vis, 'Landsat 8 Cloud-Free Composite')
m

Total Landsat 8 images available for 2020: 83


Map(center=[7.625000281267503, -58.02523831094688], controls=(WidgetControl(options=['position', 'transparent_…

### Shows a annual composite cloudless image and the areas of the image that have fewer/greater amounts of clouds

In [15]:
# 1. Start with your collection
landsat_collection = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .filterBounds(aoi_polygon)
    .filterDate(start_date, end_date)
)

# 1. Define the cloud mask function
def mask_landsat_clouds(image):
    # Bit 3 is Cloud Shadow, Bit 4 is Cloud
    qa = image.select('QA_PIXEL')
    
    # Both flags should be set to zero, indicating clear conditions
    # https://www.usgs.gov/landsat-missions/landsat-collection-2-quality-assessment-bands
    cloud_shadow_mask = qa.bitwiseAnd(1 << 3).eq(0)
    cloud_mask = qa.bitwiseAnd(1 << 4).eq(0)

    # Combine masks (pixel must be free of both cloud and shadow)
    mask = cloud_shadow_mask.And(cloud_mask)
    
    # Return the masked image, retaining only the clean pixels
    return image.updateMask(mask)

# 2. Applies scaling factors definition
def apply_scale_factors(image):
    optical_bands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
    return image.addBands(optical_bands, None, True).addBands(
        thermal_bands, None, True
    )

# 3. MAP the scaling function while it is STILL a collection 
scaled_collection = landsat_collection.map(mask_landsat_clouds).map(apply_scale_factors)

# 4. NOW reduce the scaled collection to a single composite image and clip it
landsat_composite = scaled_collection.median().clip(aoi_polygon)

# Landsat 8 True Color Bands visualization parameters
landsat_vis = {
    'min': 0.0,
    'max': 0.3,
    'bands': ['SR_B4', 'SR_B3', 'SR_B2']
}
Map = geemap.Map()
# 5. Add the final COMPOSITE IMAGE to the map (not the collection)
Map.addLayer(landsat_composite, landsat_vis, 'Landsat 8 Cloud-Free Composite')
clear_count = scaled_collection.select('SR_B4').reduce(ee.Reducer.count())
Map.addLayer(clear_count, {'min': 0, 'max': 30, 'palette': ['red', 'yellow', 'green']}, 'Clear obs per pixel')
Map.centerObject(aoi_polygon, 7)

# 6. Call the Map object at the end so it renders in your notebook
Map

Map(center=[7.625000281267503, -58.02523831094688], controls=(WidgetControl(options=['position', 'transparent_…

### Without cloud mask and just median composite

In [16]:
newMap = geemap.Map()

scaled_collection = landsat_collection.map(apply_scale_factors)

# 4. NOW reduce the scaled collection to a single composite image and clip it
landsat_scaled = scaled_collection.median().clip(aoi_polygon)

newMap.addLayer(landsat_scaled, landsat_vis, 'Landsat 8 Cloud-Free Composite')
clear_count = scaled_collection.select('SR_B4').reduce(ee.Reducer.count())
newMap.addLayer(clear_count, {'min': 0, 'max': 30, 'palette': ['red', 'yellow', 'green']}, 'Clear obs per pixel')
newMap.centerObject(aoi_polygon, 7)

newMap

Map(center=[7.625000281267503, -58.02523831094688], controls=(WidgetControl(options=['position', 'transparent_…

### Testing resampling in GEE (looks like it generates a poor result) 

Nearest neighbor resampling will be part of the native python process after exporting to GEE

In [27]:
landsat_10m = (
    landsat_collection.map(mask_l8_clouds)
        .map(add_ndvi)
        .map(lambda img: img.resample())
        .qualityMosaic('NDVI').clip(aoi_polygon)
)

mm = geemap.Map()
mm.addLayer(landsat_10m, landsat_vis, 'Landsat 8 10m')
mm.addLayer(landsat_composite, landsat_vis, 'Landsat 8 Composite')
mm.centerObject(aoi_polygon, 7)
mm

Map(center=[7.625000281267503, -58.02523831094688], controls=(WidgetControl(options=['position', 'transparent_…